# 03 Split Dataset

Split the cleaned merged original PPE dataset into train/validation folders before any augmentation is applied.


## Purpose

This notebook reads the split settings from `configs/dataset_config.yaml`, copies matched image-label pairs from `data/master_original`, and writes the generated split folders under `data/splits_original`.

When `use_external_test_set: true`, the test split is treated as an already-held-out external test set created by Notebook 01. In that mode, this notebook only generates train and validation splits and leaves `data/splits_original/test` unchanged.


## Setup


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the v2_pipeline root from the current working directory."""
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested
    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

from src.dataset.split_dataset import find_image_label_pairs, split_dataset

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

## Load Configuration


In [2]:
config_path = V2_ROOT / "configs" / "dataset_config.yaml"
with config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)

master_original_dir = V2_ROOT / dataset_config["master_original_dir"]
master_images_dir = master_original_dir / "images"
master_labels_dir = master_original_dir / "labels"
splits_original_dir = V2_ROOT / dataset_config["splits_original_dir"]

use_external_test_set = bool(dataset_config.get("use_external_test_set", True))
train_ratio = float(dataset_config["split"]["train"])
val_ratio = float(dataset_config["split"]["val"])
test_ratio = float(dataset_config["split"]["test"])
random_seed = int(dataset_config.get("random_seed", 42))

config_summary = pd.DataFrame(
    [
        {"setting": "use_external_test_set", "value": use_external_test_set},
        {"setting": "split.train", "value": train_ratio},
        {"setting": "split.val", "value": val_ratio},
        {"setting": "split.test", "value": test_ratio},
        {"setting": "random_seed", "value": random_seed},
    ]
)
display(config_summary)

,setting,value
0,use_external_test_set,True
1,split.train,0.8
2,split.val,0.2
3,split.test,0.0
4,random_seed,42


## Split Strategy


In [3]:
if use_external_test_set:
    print("Strategy: split master_original into train/val only.")
    print("Test split: preserve existing external test files from Notebook 01.")
    if test_ratio != 0.0:
        raise ValueError("use_external_test_set=True requires split.test to be 0.00")
else:
    print("Strategy: split master_original into train/val/test.")

print(f"Master images: {master_images_dir}")
print(f"Master labels: {master_labels_dir}")
print(f"Output splits: {splits_original_dir}")

Strategy: split master_original into train/val only.
Test split: preserve existing external test files from Notebook 01.
Master images: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\master_original\images
Master labels: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\master_original\labels
Output splits: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\data\splits_original


## Check Master Pairs


In [4]:
pairs = find_image_label_pairs(master_images_dir, master_labels_dir)
print(f"Found {len(pairs)} matched original image-label pairs.")

pair_preview = pd.DataFrame(
    [
        {"image": image_path.name, "label": label_path.name}
        for image_path, label_path in pairs[:10]
    ]
)
display(pair_preview)

Found 475 matched original image-label pairs.


,image,label
0,ppe_00001.png,ppe_00001.txt
1,ppe_00002.png,ppe_00002.txt
2,ppe_00003.png,ppe_00003.txt
3,ppe_00004.png,ppe_00004.txt
4,ppe_00005.png,ppe_00005.txt
5,ppe_00006.png,ppe_00006.txt
6,ppe_00007.png,ppe_00007.txt
7,ppe_00008.png,ppe_00008.txt
8,ppe_00009.png,ppe_00009.txt
9,ppe_00010.png,ppe_00010.txt


## Run Split

By default this cell refuses to overwrite existing generated train/validation files. Set `overwrite_existing_splits = True` only when intentionally regenerating the split.


In [5]:
overwrite_existing_splits = True

summary = split_dataset(
    master_images_dir=master_images_dir,
    master_labels_dir=master_labels_dir,
    output_dir=splits_original_dir,
    train_ratio=train_ratio,
    val_ratio=val_ratio,
    test_ratio=test_ratio,
    seed=random_seed,
    use_external_test_set=use_external_test_set,
    overwrite=overwrite_existing_splits,
)

summary_df = pd.DataFrame(summary.values())[
    ["split", "num_images", "num_labels", "source", "notes"]
]
display(summary_df)

if use_external_test_set and summary["test"]["num_images"] == 0:
    print("Warning: external test set is enabled, but no test images were found.")

,split,num_images,num_labels,source,notes
0,train,380,380,master_original,generated by notebook 03
1,val,95,95,master_original,generated by notebook 03
2,test,50,50,external_test_set,preserved from notebook 01


## Save Summary


In [6]:
summary_path = V2_ROOT / "reports" / "validation" / "split_summary.csv"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(summary_path, index=False)
summary_path

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline/reports/validation/split_summary.csv')

## Final Counts


In [7]:
final_counts = pd.DataFrame(
    [
        {
            "split": split_name,
            "images": row["num_images"],
            "labels": row["num_labels"],
            "source": row["source"],
            "notes": row["notes"],
        }
        for split_name, row in summary.items()
    ]
)
display(final_counts)

for split_name, row in summary.items():
    print(f"{split_name}: {row['num_images']} images, {row['num_labels']} labels")

,split,images,labels,source,notes
0,train,380,380,master_original,generated by notebook 03
1,val,95,95,master_original,generated by notebook 03
2,test,50,50,external_test_set,preserved from notebook 01


train: 380 images, 380 labels
val: 95 images, 95 labels
test: 50 images, 50 labels


## Checklist Before Notebook 04

- Train and validation folders contain matching image and label counts.
- If external test is enabled, the test folder was preserved from Notebook 01.
- `reports/validation/split_summary.csv` has been saved.
- No augmentation has been run yet.
- Continue to Notebook 04 only after reviewing the split counts.
